# Task 1.1 — Data Preparation and Validation Pipeline
**Unsupervised Learning Project 2025/2026 — Hotel Booking Demand**

## RQ1 — Segmentation Question

**Question:** Can we identify distinct booking behaviour profiles among hotel guests, based only on information available at the time of booking, that are meaningful for hotel revenue management and operations?

**Unit of analysis:** Each row in the dataset represents one booking record, not guests (the same guest can appear in multiple rows).

Clustering is appropriate here because there are no predefined guest segments (we are in an unsupervised setting). The goal is to discover latent groups of similar bookings (lead time, stay duration, guest composition, channel, etc.), not to predict a known outcome. To avoid leakage, we use only features known at or before booking; outcome variables (such as is_canceled and reservation_status) are excluded from the clustering inputs.

## Data Documentation

- **Source:** Course release v1 — `hotel_bookings_course_release_v1.csv`
- **Original:** Kaggle — [jessemostipak/hotel-booking-demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand/data)
- **Reference:** António, N., de Almeida, A., & Nunes, L. (2019). Hotel booking demand datasets. *Data in Brief*, 22, 41–49. https://doi.org/10.1016/j.dib.2018.11.126
- **License:** CC0 1.0 Public Domain (as stated in the course release)
- **What each row represents:** One hotel booking record (one stay or cancellation)
- **Time span:** Arrivals between July 2015 and August 2017, across two hotels (City Hotel and Resort Hotel)
- **Known data quality issues:**
  - `children`: 4 NaN values
  - `country`: ~0.4% missing values
  - `agent` / `company`: NULL encoded as the string `"NULL"` (not a proper NaN)
  - Some rows have 0 adults + 0 children + 0 babies (impossible — data entry error)
  - `adr` has at least one negative value (data entry error)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

SEED = 12345
np.random.seed(SEED)

df_raw = pd.read_csv('hotel_bookings_course_release_v1.csv')
print('Shape:', df_raw.shape)
df_raw.head(3)

Shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


## Leakage Control & Feature Exclusions

Clustering inputs must only use variables available **at booking time**. The following are excluded:

| Variable | Reason |
|---|---|
| `is_canceled` | Outcome variable — unknown at booking time |
| `reservation_status` | Post-event status — realised after the stay |
| `reservation_status_date` | Date of post-event status — post-arrival |
| `agent` | High-cardinality ID-like field (hundreds of numeric codes); no meaningful encoding |
| `company` | High-cardinality ID-like field; >90% NULL |

The outcome variables (`is_canceled`, `reservation_status`) are retained separately for **post-hoc profiling only**.

In [ ]:
EXCLUDED = ['is_canceled', 'reservation_status', 'reservation_status_date', 'agent', 'company']

# Keep outcome vars for post-hoc profiling
df_labels = df_raw[['is_canceled', 'reservation_status']].copy()

df = df_raw.drop(columns=EXCLUDED).copy()
print('Shape after exclusions:', df.shape)

Shape after exclusions: (119390, 27)


## Data Quality Fixes

In [ ]:
# Fix 1: children NaN -> 0 (no child is the correct interpretation for missing)
df['children'] = df['children'].fillna(0).astype(int)

# Fix 2: remove rows with 0 adults + 0 children + 0 babies (impossible)
zero_guests = (df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)
print(f'Zero-guest rows removed: {zero_guests.sum()}')
df = df[~zero_guests].copy()

# Fix 3: negative ADR -> NaN (will be median-imputed in pipeline)
neg_adr = df['adr'] < 0
print(f'Negative ADR values set to NaN: {neg_adr.sum()}')
df.loc[neg_adr, 'adr'] = np.nan

# Align profiling labels
df_labels = df_labels.loc[df.index]

print('Working dataset shape:', df.shape)

Zero-guest rows removed: 180
Negative ADR values set to NaN: 1
Working dataset shape: (119210, 27)


## Feature Set Definition

In [ ]:
NUMERICAL = df.select_dtypes(include=[np.number]).columns.tolist()
CATEGORICAL = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f'Numerical features  ({len(NUMERICAL)}): {NUMERICAL}')
print(f'Categorical features ({len(CATEGORICAL)}): {CATEGORICAL}')

Numerical features  (17): ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
Categorical features (10): ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']


## Missingness Report

In [ ]:
df_features = df[NUMERICAL + CATEGORICAL].copy()

print('=== Missingness — Numerical features ===')
miss_num = df_features[NUMERICAL].isnull().sum()
miss_num_pct = (miss_num / len(df_features) * 100).round(3)
miss_num_df = pd.DataFrame({'n_missing': miss_num, 'pct_missing': miss_num_pct})
print(miss_num_df[miss_num_df['n_missing'] > 0].to_string())
print('(All others: 0 missing)' if (miss_num == 0).all() else '')

print()
print('=== Missingness — Categorical features ===')
miss_cat = df_features[CATEGORICAL].isnull().sum()
miss_cat_pct = (miss_cat / len(df_features) * 100).round(3)
miss_cat_df = pd.DataFrame({'n_missing': miss_cat, 'pct_missing': miss_cat_pct})
print(miss_cat_df[miss_cat_df['n_missing'] > 0].to_string())
if (miss_cat == 0).all():
    print('(All others: 0 missing)')

=== Missingness — Numerical features ===
     n_missing  pct_missing
adr          1        0.001


=== Missingness — Categorical features ===
         n_missing  pct_missing
country        478        0.401


## Outlier Report (Numerical — IQR method)

In [15]:
print('=== Outliers — Numerical features (IQR rule: outside Q1-1.5*IQR / Q3+1.5*IQR) ===')
rows = []
for col in NUMERICAL:
    s = df_features[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((s < lo) | (s > hi)).sum()
    rows.append({'feature': col, 'lower_fence': round(lo,2), 'upper_fence': round(hi,2),
                 'n_outliers': n_out, 'pct': round(n_out/len(s)*100, 2)})

outlier_df = pd.DataFrame(rows).set_index('feature')
print(outlier_df.to_string())

print(f'\n=== Outliers — Categorical features ===')
print('Not applicable (no numerical range). Rare categories handled via frequency threshold in pipeline.')

=== Outliers — Numerical features (IQR rule: outside Q1-1.5*IQR / Q3+1.5*IQR) ===
                                lower_fence  upper_fence  n_outliers    pct
feature                                                                    
lead_time                           -196.50       375.50        2981   2.50
arrival_date_year                   2014.50      2018.50           0   0.00
arrival_date_week_number             -17.00        71.00           0   0.00
arrival_date_day_of_month            -14.50        45.50           0   0.00
stays_in_weekend_nights               -3.00         5.00         258   0.22
stays_in_week_nights                  -2.00         6.00        3330   2.79
adults                                 2.00         2.00       29530  24.77
children                               0.00         0.00        8590   7.21
babies                                 0.00         0.00         917   0.77
is_repeated_guest                      0.00         0.00        3755   3.15
previo

## Preprocessing Pipeline

**Numerical:** median imputation (robust to outliers/skew) → StandardScaler (z-score normalization)

**Categorical:** mode imputation → rare categories (< 1% frequency) grouped into `"Other"` → OneHotEncoder with `handle_unknown='ignore'` (unseen categories at inference map to all-zero vector)

**Implied distance/metric:** After this pipeline the representation is a dense numeric matrix with standardised numerical columns and binary OHE columns. The natural and implied distance is **Euclidean distance in this standardised mixed space**, which is compatible with k-means and Ward hierarchical clustering.

In [16]:
# Group rare categories (< 1% frequency) into 'Other' before OHE
MIN_FREQ = 0.01
for col in CATEGORICAL:
    freq = df_features[col].value_counts(normalize=True)
    rare = freq[freq < MIN_FREQ].index
    if len(rare) > 0:
        df_features[col] = df_features[col].apply(lambda x: 'Other' if x in rare else x)
        print(f'{col}: {len(rare)} rare categories grouped into "Other"')

meal: 2 rare categories grouped into "Other"
country: 163 rare categories grouped into "Other"
market_segment: 3 rare categories grouped into "Other"
distribution_channel: 2 rare categories grouped into "Other"
reserved_room_type: 4 rare categories grouped into "Other"
assigned_room_type: 4 rare categories grouped into "Other"
deposit_type: 1 rare categories grouped into "Other"
customer_type: 1 rare categories grouped into "Other"


In [17]:
numerical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline,  NUMERICAL),
    ('cat', categorical_pipeline, CATEGORICAL),
], remainder='drop')

X = preprocessor.fit_transform(df_features)

ohe_names = (
    preprocessor
    .named_transformers_['cat']
    .named_steps['encode']
    .get_feature_names_out(CATEGORICAL)
    .tolist()
)
feature_names = NUMERICAL + ohe_names
X_df = pd.DataFrame(X, columns=feature_names, index=df_features.index)

print(f'Final matrix shape : {X_df.shape}')
print(f'  - Numerical dims : {len(NUMERICAL)}')
print(f'  - OHE dims       : {len(ohe_names)}')
print(f'  - Total samples  : {X_df.shape[0]:,}')
print(f'NaN in output      : {X_df.isnull().sum().sum()}')

Final matrix shape : (119210, 81)
  - Numerical dims : 17
  - OHE dims       : 64
  - Total samples  : 119,210
NaN in output      : 0


## Final Feature Set Summary

| | |
|---|---|
| **Samples** | see output above |
| **Numerical features** | 17 (standardised, median-imputed) |
| **Categorical features** | 10 (one-hot encoded, rare→Other) |
| **Total dimensions after OHE** | see output above |
| **Implied distance** | Euclidean in standardised mixed space |
| **Compatible algorithms** | k-means, MiniBatch k-means, Ward hierarchical |
| **Excluded (leakage)** | `is_canceled`, `reservation_status`, `reservation_status_date` |
| **Excluded (ID-like)** | `agent`, `company` |
| **Retained for profiling only** | `is_canceled`, `reservation_status` |

# Task 1.2 - Learn a baseline clustering model

In [ ]:
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score

kvalues = range(3, 8) # Test k = 3 to 7 clusters (5 total)
SEEDS = range(5)  # Multiple seeds for stability check
np.random.seed(SEEDS)
sample_indices = np.random.choice(X_df.index, size=int(len(X_df)*0.1), replace=False) # Use 10% sample for faster clustering and metric computation
X_sample = X_df.loc[sample_indices]

results = []
for k in kvalues:
    print(f'\n=== Clustering with k={k} ===')
    for seed in SEEDS:
        ward_model = AgglomerativeClustering(n_clusters=k, linkage='ward')
        ward_labels = ward_model.fit_predict(X_sample)
        ward_s_score = silhouette_score(X_sample, ward_labels)
        results.append({'clustering': 'Ward', 'k': k, 'seed': seed, 'silhouette': ward_s_score})
        ward_db_score = davies_bouldin_score(X_sample, ward_labels)
        results.append({'clustering': 'Ward', 'k': k, 'seed': seed, 'davies_bouldin': ward_db_score})
        ward_ch_score = calinski_harabasz_score(X_sample, ward_labels)
        results.append({'clustering': 'Ward', 'k': k, 'seed': seed, 'calinski_harabasz': ward_ch_score})
        print(f'Ward k={k} seed={seed} -> silhouette={ward_s_score:.4f}, davies_bouldin={ward_db_score:.4f}, calinski_harabasz={ward_ch_score:.4f}')

        kmeans = KMeans(n_clusters=k, random_state=seed)
        labels = kmeans.fit_predict(X_sample)
        s_score = silhouette_score(X_sample, labels)
        results.append({'clustering': 'KMeans', 'k': k, 'seed': seed, 'silhouette': s_score})
        db_score = davies_bouldin_score(X_sample, labels)
        results.append({'clustering': 'KMeans', 'k': k, 'seed': seed, 'davies_bouldin': db_score})
        ch_score = calinski_harabasz_score(X_sample, labels)
        results.append({'clustering': 'KMeans', 'k': k, 'seed': seed, 'calinski_harabasz': ch_score})
        print(f'KMeans k={k} seed={seed} -> silhouette={s_score:.4f}, davies_bouldin={db_score:.4f}, calinski_harabasz={ch_score:.4f}')

        mbk = MiniBatchKMeans(n_clusters=k, random_state=seed)
        mb_labels = mbk.fit_predict(X_sample)
        mb_s_score = silhouette_score(X_sample, mb_labels)
        results.append({'clustering': 'MiniBatchKMeans', 'k': k, 'seed': seed, 'silhouette': mb_s_score})
        mb_db_score = davies_bouldin_score(X_sample, mb_labels)
        results.append({'clustering': 'MiniBatchKMeans', 'k': k, 'seed': seed, 'davies_bouldin': mb_db_score})
        mb_ch_score = calinski_harabasz_score(X_sample, mb_labels)
        results.append({'clustering': 'MiniBatchKMeans', 'k': k, 'seed': seed, 'calinski_harabasz': mb_ch_score})
        print(f'MiniBatchKMeans k={k} seed={seed} -> silhouette={mb_s_score:.4f}, davies_bouldin={mb_db_score:.4f}, calinski_harabasz={mb_ch_score:.4f}')
        results_df = pd.DataFrame(results)


=== Clustering with k=3 ===
Ward k=3 seed=0 -> silhouette=0.3800, davies_bouldin=1.1072, calinski_harabasz=695.6297
KMeans k=3 seed=0 -> silhouette=0.0682, davies_bouldin=3.0691, calinski_harabasz=685.9494
MiniBatchKMeans k=3 seed=0 -> silhouette=0.1524, davies_bouldin=2.6459, calinski_harabasz=709.0509
Ward k=3 seed=1 -> silhouette=0.3800, davies_bouldin=1.1072, calinski_harabasz=695.6297
KMeans k=3 seed=1 -> silhouette=0.0748, davies_bouldin=2.9778, calinski_harabasz=676.4603
MiniBatchKMeans k=3 seed=1 -> silhouette=0.0663, davies_bouldin=3.3165, calinski_harabasz=613.9397
Ward k=3 seed=2 -> silhouette=0.3800, davies_bouldin=1.1072, calinski_harabasz=695.6297
KMeans k=3 seed=2 -> silhouette=0.0681, davies_bouldin=3.0692, calinski_harabasz=685.9489
MiniBatchKMeans k=3 seed=2 -> silhouette=0.0995, davies_bouldin=2.7743, calinski_harabasz=569.4335
Ward k=3 seed=3 -> silhouette=0.3800, davies_bouldin=1.1072, calinski_harabasz=695.6297
KMeans k=3 seed=3 -> silhouette=0.0675, davies_bould

In [34]:
print(f'\nBest silhouette score: {results_df.loc[results_df["silhouette"].idxmax()]["silhouette"]:.4f} with seed {results_df.loc[results_df["silhouette"].idxmax()]["seed"]}, clustering method: {results_df.loc[results_df["silhouette"].idxmax()]["clustering"]} and k = {results_df.loc[results_df["silhouette"].idxmax()]["k"]}')
print(f'\nBest davies_bouldin score: {results_df.loc[results_df["davies_bouldin"].idxmin()]["davies_bouldin"]:.4f} with seed {results_df.loc[results_df["davies_bouldin"].idxmin()]["seed"]}, clustering method: {results_df.loc[results_df["davies_bouldin"].idxmin()]["clustering"]} and k = {results_df.loc[results_df["davies_bouldin"].idxmin()]["k"]}')
print(f'\nBest calinski_harabasz score: {results_df.loc[results_df["calinski_harabasz"].idxmax()]["calinski_harabasz"]:.4f} with seed {results_df.loc[results_df["calinski_harabasz"].idxmax()]["seed"]}, clustering method: {results_df.loc[results_df["calinski_harabasz"].idxmax()]["clustering"]} and k = {results_df.loc[results_df["calinski_harabasz"].idxmax()]["k"]}')


Best silhouette score: 0.3800 with seed 0, clustering method: Ward and k = 3

Best davies_bouldin score: 1.1072 with seed 0, clustering method: Ward and k = 3

Best calinski_harabasz score: 737.1805 with seed 3, clustering method: KMeans and k = 4


In [ ]:
# Build DataFrame and collapse to one row per (clustering, k, seed) with all three metrics
results_df = pd.DataFrame(results)
df_runs = results_df.groupby(['clustering', 'k', 'seed']).max().reset_index()
metrics = ['silhouette', 'davies_bouldin', 'calinski_harabasz']
agg = df_runs.groupby(['clustering', 'k'])[metrics].agg(['mean', 'std']).round(4)

# Display table: mean ± std per (clustering, k)
print('\n=== Clustering results (mean ± std over 5 seeds) ===')
table_rows = []
for (clustering, k), row in agg.iterrows():
    r = {'clustering': clustering, 'k': k}
    for m in metrics:
        mu, sig = row[(m, 'mean')], row[(m, 'std')]
        r[m] = f'{mu:.4f} ± {sig:.4f}'
    table_rows.append(r)
table_df = pd.DataFrame(table_rows).set_index(['clustering', 'k'])
display(table_df)


=== Clustering results (mean ± std over 5 seeds) ===


silhouette   davies_bouldin   calinski_harabasz
clustering      k                                                      
KMeans          3  0.0710 ± 0.0042  3.0859 ± 0.0969  666.9123 ± 37.5072
                4  0.0708 ± 0.0041  2.7913 ± 0.1020  644.9874 ± 57.2294
                5  0.0758 ± 0.0066  2.5412 ± 0.0861  656.4969 ± 44.7375
                6  0.0771 ± 0.0040  2.3719 ± 0.0914  656.2548 ± 13.0697
                7  0.0745 ± 0.0088  2.2770 ± 0.2083  637.4871 ± 48.3607
MiniBatchKMeans 3  0.0956 ± 0.0359  2.9222 ± 0.2898  623.1036 ± 63.1975
                4  0.0673 ± 0.0354  2.9826 ± 0.3227  564.8119 ± 53.2900
                5  0.0486 ± 0.0172  2.8814 ± 0.2063  536.3730 ± 73.6746
                6  0.0552 ± 0.0171  2.6782 ± 0.1449  551.2391 ± 38.7078
                7  0.0433 ± 0.0206  2.6982 ± 0.1427  516.9253 ± 50.7613
Ward            3  0.3800 ± 0.0000  1.1072 ± 0.0000   695.6297 ± 0.0000
                4  0.2714 ± 0.0000  1.5534 ± 0.0000   689.4269 ± 0.0000
                5  0.0647 ± 0.0000  2.2747 ± 0.0000   691.2792 ± 0.0000
                6  0.0738 ± 0.0000  1.9679 ± 0.0000   700.7893 ± 0.0000
                7  0.0808 ± 0.0000  1.9598 ± 0.0000   707.2571 ± 0.0000